### Imports


In [1]:
# Imports - basic
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# Configs
from config.config import SAE_Config
from config.config import CNN_Config

# Classes
from dataloader.load_data import LoadData
from model.cnn import ExplainableCNN
from model.sae import SAE
from trainer.trainer import Trainer

# Utils
from utils.hooks import Hooks

### Dataset and transforms


In [2]:
# Define transforms
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Load Dataset
data = LoadData(transform=transform)
training_data, testing_data = data.load_data("CIFAR10")

# Data Loaders
train_loader = data.data_loaders(training_data, "train")
test_loader = data.data_loaders(testing_data, "test")

# Example visualization
examples = iter(train_loader)
samples, labels = next(examples)
print(samples.shape, labels.shape)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([10, 3, 32, 32]) torch.Size([10])


### Model Configs


In [3]:
# Import configs
sae_model_arguments = SAE_Config['model_args']
sae_model_arguments

{'input_dims': 10,
 'hidden_dims': 25,
 'epochs': 2,
 'batch_size': 1,
 'learning_rate': 0.01}

### Model


In [4]:
sae = SAE(sae_model_arguments['input_dims'], nn.ReLU, sae_model_arguments['hidden_dims'])
sae

SAE(
  (encoder): Linear(in_features=10, out_features=25, bias=True)
  (relu): ReLU()
  (decoder): Linear(in_features=25, out_features=10, bias=True)
)

In [5]:
# Define the criterion and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(sae.parameters(), lr=sae_model_arguments['learning_rate'])

### Feature extraction


In [ ]:
experiment_1_configs = CNN_Config['model_args']
experiment_1_configs

In [ ]:
cnn_model = ExplainableCNN(input_size=experiment_1_configs['input_size'], hidden_layers=experiment_1_configs['hidden_layers'], 
                       num_classes=experiment_1_configs['num_classes'], activation=experiment_1_configs['activation'], 
                       normalization=experiment_1_configs['norm_layer'], max_pool=experiment_1_configs['max_pool'], 
                       drop_prob=experiment_1_configs['drop_prob'])

cnn_model.load_state_dict(torch.load('model_weights.pth'))
cnn_model.eval()

In [ ]:
feature_collector = Hooks(cnn_model, ['conv5'])

In [ ]:
for i in range(5):
    data, _ = testing_data[i]
    data.unsqueeze_(0)
    output = cnn_model(data)
    feature_collector.collect_features('conv5')

feature_collector.write_features(filepath="data/features.py")

len(feature_collector.feature_vectors)

In [ ]:
feature_collector.remove()

### Training


In [ ]:
# Training


['tensor([-1.4766e+00,  1.0154e+01,  1.4303e+00, -8.8727e-02, -2.4639e-01,\n         7.3128e+00, -8.9850e-01,  8.6636e+00,  1.3284e-01,  5.6279e+00,\n        -5.0312e-01, -1.6701e-01,  8.2384e-01,  2.3831e-01, -4.5241e-01,\n         1.6835e-01,  6.3635e+00, -3.5305e-01, -8.9296e-01, -6.6255e+00,\n        -1.3362e-02,  4.7488e-01,  1.3128e+00, -7.6238e-01,  3.5324e+00,\n        -5.2451e-01,  4.3567e-01, -4.7409e-01, -1.1375e+00,  5.0244e-01,\n         4.2725e-02,  6.0919e-01,  3.8373e+00,  2.0485e+00, -1.4344e+00,\n        -2.9617e+00, -2.4349e-02, -3.6687e-01, -8.3861e-01, -5.4838e-01,\n        -1.6330e+00,  1.6401e+01,  4.7617e-01, -1.8859e-01, -1.1928e+00,\n        -9.0876e-02, -4.6585e+00, -8.2397e-01, -1.8597e-01,  4.9733e+00,\n         5.7416e-01,  1.0240e+00,  2.5183e+00,  1.9218e+00,  7.8550e-01,\n         6.5089e+00,  1.0876e+01,  1.1470e+00,  7.4764e+00, -1.2677e+00,\n         7.8448e-01, -1.2642e+00,  5.0474e-01, -6.6839e-01]', '\ntensor([-2.1762e+00,  1.4557e+01,  2.1486e+00

In [ ]:
# Saving

### Hooks


In [ ]:
# Hooks on the SAE
sae_model = SAE(sae_model_arguments['input_dims'], nn.ReLU, sae_model_arguments['hidden_dims'])

collector = Hooks(model=sae_model, module_names=['encoder'])
activations = collector.activations['encoder']
